In [31]:
import pandas as pd
from openai import OpenAI
import pdfplumber
from pathlib import Path
import os
import pdfplumber
from collections import defaultdict
import numpy as np

In [32]:
path = Path("sample_documents/cactus/Show packing slip.pdf")

def group_words_by_line(words, tol=3):
    lines = defaultdict(list)
    for word in words:
        placed = False
        for top_key in list(lines.keys()):
            if abs(word['top'] - top_key) <= tol:
                lines[top_key].append(word)
                placed = True
                break
        if not placed:
            lines[word['top']].append(word)
    # Sort lines by their vertical position (top)
    return [sorted(lines[key], key=lambda w: w['x0']) for key in sorted(lines.keys())]

def infer_columns(line_words, gap_threshold=5):
    columns = []
    current_col = []
    prev_x1 = None

    for word in line_words:
        if prev_x1 is None:
            current_col.append(word['text'])
        else:
            gap = word['x0'] - prev_x1
            if gap > gap_threshold:
                # big gap => new column
                columns.append(" ".join(current_col))
                current_col = [word['text']]
            else:
                current_col.append(word['text'])
        prev_x1 = word['x1']
    if current_col:
        columns.append(" ".join(current_col))
    return columns

def group_words_by_line(words, tol=3):
    lines = defaultdict(list)
    for w in words:
        placed = False
        for y in lines.keys():
            if abs(w['top'] - y) <= tol:
                lines[y].append(w)
                placed = True
                break
        if not placed:
            lines[w['top']].append(w)
    # Sort lines by vertical position
    return [sorted(lines[y], key=lambda w: w['x0']) for y in sorted(lines.keys())]

def reconstruct_with_leading_spaces(page, max_space=3, min_gap=5, space_per_10pt=1):
    words = page.extract_words()
    lines = group_words_by_line(words)
    lines_text = []

    # Find min x0 for reference (left margin)
    min_x0 = min(w['x0'] for w in words)

    for line in lines:
        line_str = ""

        # Calculate leading spaces based on first word x0 relative to min_x0
        leading_gap = line[0]['x0'] - min_x0
        leading_spaces = int(leading_gap / 10) * space_per_10pt
        line_str += " " * leading_spaces

        prev_x1 = None
        for w in line:
            if prev_x1 is None:
                line_str += w['text']
            else:
                gap = w['x0'] - prev_x1
                if gap < min_gap:
                    spaces = 1
                else:
                    spaces = min(max_space, int(gap // min_gap))
                line_str += " " * spaces + w['text']
            prev_x1 = w['x1']

        lines_text.append(line_str)

    return "\n".join(lines_text)


with pdfplumber.open(path) as pdf:
    page = pdf.pages[0]
    text_with_leading_spaces = reconstruct_with_leading_spaces(page)
    print(text_with_leading_spaces)


     Måkestad Engros AS   Følgeseddel   01786141
                       Salgsordre   3431217   Side   1 av 2
                       Kundenummer   8034313   Opprinnelse   Nettbutikk
                       Forsendelsesdato   15.09.2025   Pulje   Norm
                       Ordredato   13.09.2025   Rute   R05-1
                                        Tur
                                        Posisjon
   Kunde   Leveringsadresse
   AVVENTURA AS   CACTUS GALLERIET KL 08-10
   Torgallmenningen 8   TLF 91179466, TORGALLMENNINGEN 8
   5014 BERGEN   5014 BERGEN
   NOR   NOR
Deres referanse   Deres rekvisisjon   Kontakt/tlf   Bestilt av   8034313
             PO1429546977775   Cactus Galleriet
                                       5014 BERGEN
Leveringsmåte   Fraktet av   Salgsansvarlig
Bystasjonen M/O/F   Carrier
Leveringsbetingelser   Bekreftet forsendelsesdato   Vår ref.   Transit Id
Fritt Levert   15.09.2025   Autogodkjent
            Leverandør   Pakning   Levert
Pos Varenummer   Varenr  

In [48]:
import json
import requests
from pathlib import Path

API_KEY = "sk-22075a23b9844efb83373bf08512dea0"
API_URL = "https://api.deepseek.com/v1"  # Adjust if different
path = Path("sample_documents/cactus/Show packing slip.pdf")


client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com")

keys = [
    "PO Number",
    "Order Date",
    "Delivery Date",
    "Vendor",
    "SKU",
    "Product Name",
    "Quantity",
    "Units",
    "Price",
]

with pdfplumber.open(path) as pdf:
    page = pdf.pages[0]
    text_with_leading_spaces = reconstruct_with_leading_spaces(page)

system_prompt = (
    "You are a helpful assistant that extracts structured data from pdfs. "
    "You will be provided with the text content of a pdf, and you need to "
    "extract specific fields for each line item in the pdf and return them ONLY in JSON format with the specified keys."
    "Make your best guess based on the text provided - all values must be filled."
)

user_prompt = f"""
Extract the following fields from the packing slip text: {', '.join(keys)}.
Here is the packing slip text:
{text_with_leading_spaces}

Respond ONLY with the JSON object with all the line items and their key values (no explanations or extra text).
"""

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    response_format={
        'type': "json_object"
    },  # only if the client supports this
    temperature=0
)
result = response.choices[0].message.content

print(json.loads(result))


{'PO Number': 'PO1429546977775', 'Order Date': '13.09.2025', 'Delivery Date': '15.09.2025', 'Vendor': 'Måkestad Engros AS', 'Line Items': [{'SKU': '6668966', 'Product Name': 'CACTUS SOS POSE LITEN HVIT 1000STK', 'Quantity': '1.00', 'Units': 'STK', 'Price': '499.90'}, {'SKU': '6700835', 'Product Name': 'FATLAND KJØTTDEIG 21% CA2KG KG SALG', 'Quantity': '48.00', 'Units': 'KG', 'Price': '114.07'}, {'SKU': '1456003', 'Product Name': 'MISSION TORTILLA HVETE 30CM MF 4X18STK', 'Quantity': '4.00', 'Units': 'KRT', 'Price': '342.99'}, {'SKU': '2474955', 'Product Name': 'Q-MEIERIET Q LETTRØMME 2,5KG', 'Quantity': '15.00', 'Units': 'STK', 'Price': '123.10'}, {'SKU': '1189737', 'Product Name': 'SANTA MARIA PINTO BEANS 2550G', 'Quantity': '24.00', 'Units': 'STK', 'Price': '80.74'}, {'SKU': '2066009', 'Product Name': 'SVEINES MAISCORN 3KG', 'Quantity': '36.00', 'Units': 'STK', 'Price': '64.65'}, {'SKU': '5306949', 'Product Name': 'VESTFOLD FUGL Porsjonsfryst KYLLING LÅRKJØTT STRIMLET 2,5KG', 'Quantit

In [ ]:
print(result)

print(response.usage)

# save result to json file
out


{
    "PO Number": "PO1429546977775",
    "Order Date": "13.09.2025",
    "Delivery Date": "15.09.2025",
    "Vendor": "Måkestad Engros AS",
    "Line Items": [
        {
            "SKU": "6668966",
            "Product Name": "CACTUS SOS POSE LITEN HVIT 1000STK",
            "Quantity": "1.00",
            "Units": "STK",
            "Price": "499.90"
        },
        {
            "SKU": "6700835",
            "Product Name": "FATLAND KJØTTDEIG 21% CA2KG KG SALG",
            "Quantity": "48.00",
            "Units": "KG",
            "Price": "114.07"
        },
        {
            "SKU": "1456003",
            "Product Name": "MISSION TORTILLA HVETE 30CM MF 4X18STK",
            "Quantity": "4.00",
            "Units": "KRT",
            "Price": "342.99"
        },
        {
            "SKU": "2474955",
            "Product Name": "Q-MEIERIET Q LETTRØMME 2,5KG",
            "Quantity": "15.00",
            "Units": "STK",
            "Price": "123.10"
        },
        {
 

In [ ]:
raw_output = response.choices[0].message.content

# Optional: Validate and pretty-print JSON, otherwise save raw output
try:
    parsed_json = json.loads(raw_output)
    pretty_json = json.dumps(parsed_json, indent=2)
    
except json.JSONDecodeError:
    print("Warning: Response is not valid JSON, saving raw output.")
    pretty_json = raw_output

output_path = Path("output.json")
with open(output_path, "w") as f:
    f.write(pretty_json)

print(f"Output saved to {output_path}")
